# Media validation procedure

Interactive QA for post media before/after publishing. Complements the
deterministic pre-flight in `app/services/publishing.py`
(`validate_media_for_platform`) with a **semantic** check: a DMR vision
model (`qwen3-vl`) captions the image and the caption is compared to the
post intent.

Procedure per asset:
1. **Fetch** the media from the SocialAuto API (`/api/v1/media/view?path=`).
2. **Deterministic checks** — decodable, dimensions/ratio/size within the
   platform rules (same table as the pre-flight).
3. **Semantic check** (DMR qwen3-vl) — "describe this image"; a mismatch
   between description and intent flags the asset.
4. **Report** — JSON verdict per asset: `pass | fail | warn` + reasons.

Run: `docker exec social-jupyter papermill work/media_validation.ipynb
work/media_validation-output.ipynb -p media_paths '["2026/10/04/x.jpg"]'
-p platform instagram`

The platform rules (kept in sync with
`_PLATFORM_MEDIA_RULES` in `app/services/publishing.py`, verified 2026-10-04):

| platform | formats | min px | max px | max bytes | ratio | max count |
|---|---|---|---|---|---|---|
| instagram | jpg/png | 320 | 1440 | 8MB | 4:5–1.91:1 | 10 |
| tiktok | jpg/png/webp/mp4/mov | 360 | 4096 | 1GB | — | 35 (photos) / 1 video |
| linkedin | jpg/png/gif + PDF | 200 | 6024 | 100MB | — | 20 |
| facebook | jpg/png/gif | 200 | 4096 | 10MB | — | 10 |
| twitter | jpg/png/webp | 200 | 8192 | 5MB | — | 4 |
| threads | jpg/png/gif | 200 | 1920 | 100MB | — | 20 |

In [ ]:
# papermill parameters (tagged cell)
media_paths = ["2026/10/04/f66a441d654f4e04.jpg"]  # storage paths, not local paths
platform = "instagram"
post_id = None  # optional — used in the report only

In [ ]:
import io, json, os, urllib.parse
from pathlib import Path

import httpx
from PIL import Image

API = os.environ.get("SOCIAL_API_URL", "http://social-api:8000")
DMR_BASE = os.environ.get("DMR_BASE_URL", "http://host.docker.internal:12435/engines/v1")
DMR_MODEL = os.environ.get("DMR_MODEL", "qwen3-vl")

# Login (admin creds come from the jupyter service environment)
r = httpx.post(f"{API}/api/v1/auth/login",
               data={"username": os.environ["SOCIAL_ADMIN_EMAIL"],
                     "password": os.environ["SOCIAL_ADMIN_PASSWORD"]}, timeout=30)
TOKEN = r.json()["access_token"]
HDRS = {"Authorization": f"Bearer {TOKEN}"}
print("authenticated to", API)

In [ ]:
# Platform rules — mirror of _PLATFORM_MEDIA_RULES (keep in sync)
PLATFORM_RULES = {
    "instagram": {"required": True, "min_dim": 320, "max_dim": 1440, "max_bytes": 8*1024*1024, "ratio": (4/5, 1.91), "max_count": 10},
    "tiktok":    {"required": True, "min_dim": 360, "max_dim": 4096, "max_bytes": 1024**3, "ratio": None, "max_count": 35},
    "linkedin":  {"required": False, "min_dim": 200, "max_dim": 6024, "max_bytes": 100*1024*1024, "ratio": None, "max_count": 20},
    "facebook":  {"required": False, "min_dim": 200, "max_dim": 4096, "max_bytes": 10*1024*1024, "ratio": None, "max_count": 10},
    "twitter":   {"required": False, "min_dim": 200, "max_dim": 8192, "max_bytes": 5*1024*1024, "ratio": None, "max_count": 4},
    "threads":   {"required": False, "min_dim": 200, "max_dim": 1920, "max_bytes": 100*1024*1024, "ratio": None, "max_count": 20},
}

def fetch_media(storage_path: str) -> bytes:
    q = urllib.parse.quote(storage_path)
    r = httpx.get(f"{API}/api/v1/media/view?path={q}", headers=HDRS, timeout=60,
                  follow_redirects=True)
    r.raise_for_status()
    return r.content

In [ ]:
def deterministic_checks(data: bytes, storage_path: str) -> dict:
    reasons, ok = [], True
    rules = PLATFORM_RULES[platform]
    if len(data) < 10 * 1024:
        ok, _ = False, reasons.append(f"file only {len(data)} bytes — truncated/placeholder")
    im = Image.open(io.BytesIO(data)); im.verify()
    im = Image.open(io.BytesIO(data))
    w, h = im.size
    if w < rules["min_dim"] or h < rules["min_dim"]:
        ok = False; reasons.append(f"{w}x{h} below min {rules['min_dim']}px")
    if w > rules["max_dim"] or h > rules["max_dim"]:
        ok = False; reasons.append(f"{w}x{h} above max {rules['max_dim']}px")
    if rules["ratio"]:
        ratio = w / h
        if not (rules["ratio"][0] <= ratio <= rules["ratio"][1]):
            ok = False; reasons.append(f"ratio {ratio:.2f} outside {rules['ratio']}")
    return {"path": storage_path, "size": w, "height": h,
            "verdict": "pass" if ok else "fail", "reasons": reasons}

results = [deterministic_checks(fetch_media(p), p) for p in media_paths]
results

In [ ]:
def dmr_caption(data: bytes, question: str) -> str:
    """Ask the DMR vision model (qwen3-vl) to describe the image."""
    b64 = base64.b64encode(data).decode()
    r = httpx.post(f"{DMR_BASE}/chat/completions", timeout=180, json={
        "model": DMR_MODEL,
        "messages": [{"role": "user", "content": [
            {"type": "text", "text": question},
            {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{b64}"}},
        ]}],
    })
    r.raise_for_status()
    return r.json()["choices"][0]["message"]["content"]

for res in results:
    if res["verdict"] != "pass":
        continue
    try:
        desc = dmr_caption(fetch_media(res["path"]),
                           "Describe this image in one factual sentence.")
        res["dmr_description"] = desc.strip()
        print(res["path"], "->", res["dmr_description"])
    except Exception as exc:  # DMR/model not pulled — semantic check is best-effort
        res["dmr_description"] = None
        res["reasons"].append(f"DMR check unavailable: {exc}")
        print("DMR check skipped:", exc)
results

In [ ]:
report = {
    "platform": platform, "post_id": post_id,
    "generated_at": __import__("datetime").datetime.utcnow().isoformat() + "Z",
    "results": results,
    "all_pass": all(r["verdict"] == "pass" for r in results),
}
print(json.dumps(report, indent=2))
# Persist next to the notebook for the triage record
Path("media_validation_report.json").write_text(json.dumps(report, indent=2))